# Extract Order Book Features (Amberdata)

This script extracts the following fields:
- Best_bid
- best_ask
- spread
- spread_pct
- bid_depth_top5
- ask_depth_top5
- bid_depth_top10
- ask_depth_top10
- order_book_imbalance_top5
- order_book_imbalance_top10

An extract is created per Cryptocurrency coin:
- LTCUSDT
- BTCUSDT
- ETHUSDT
- SOLUSDT
- XRPUSDT
- DOGEUSDT
- BNBUSDT
- ADAUSDT
- LINKUSDT
- AVAXUSDT
- DOTUSDT
- BCHUSDT

In [4]:
import requests
import pandas as pd
from datetime import datetime, timedelta, UTC
import time
from pathlib import Path

API_KEY = "" # insert key here
BASE_URL = ("https://api.amberdata.com/"
            "markets/futures/order-book-snapshots")
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT"]

headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br"}

# Date Ranges
start_date = datetime(2025, 7, 1, tzinfo=UTC)
end_date = datetime.now(UTC)
chunk_days = 3
max_retries = 5
base_retry_wait = 10

# Functions
def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None

def calculate_depth(levels, top_n):
    depth = 0.0
    for level in levels[:top_n]:
        price = safe_float(level.get("price"))
        volume = safe_float(level.get("volume"))
        if (price is not Noneand volume is not None):
            depth += (price * volume)
    return depth

def calculate_imbalance(bid_depth, ask_depth):
    total_depth = bid_depth + ask_depth
    if total_depth == 0:
        return None
    return (bid_depth - ask_depth) / total_depth

def extract_orderbook_features(instrument):
    symbol = instrument.replace("USDT", "").lower()
    checkpoint_file = (f"{symbol}_orderbook_features_checkpoint.parquet")
    final_parquet_file = (f"{symbol}_orderbook_features_1m.parquet")
    final_csv_file = (f"{symbol}_orderbook_features_1m.csv")
    print("#######################################################################")
    print(f"Starting extraction for {instrument}")
    print("#######################################################################")
    all_rows = []

    # OPTIONAL: RESUME FROM CHECKPOINT
    if Path(checkpoint_file).exists():
        print(f"Loading checkpoint: {checkpoint_file}")
        checkpoint_df = pd.read_parquet(checkpoint_file)
        all_rows = (checkpoint_df.to_dict("records"))
        latest_timestamp = pd.to_datetime(checkpoint_df["exchangeTimestamp"].max(),utc=True)
        current_start = (latest_timestamp.to_pydatetime() + timedelta(seconds=1))
        print(f"Resuming {instrument} from: "f"{current_start}")
    else:
        current_start = start_date

    # MAIN LOOP FOR THIS INSTRUMENT
    while current_start < end_date:
        current_end = min(current_start + timedelta(days=chunk_days),end_date)

        print("#######################################################################")
        print(f"{instrument} chunk: "f"{current_start}"f" → "f"{current_end}")
        next_url = (f"{BASE_URL}/"f"{instrument}")
        params = {"exchange": exchange,"startDate":current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate":current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat":"iso8601"}
        page_count = 0
        total_rows = 0
        chunk_failed = False

        while next_url:
            retries = 0
            while retries < max_retries:
                try:
                    response = requests.get(next_url,headers=headers,params=params,timeout=60)
                    response.raise_for_status()
                    data = response.json()
                    break

                except requests.exceptions.RequestException as e:
                    retries += 1
                    wait_time = (base_retry_wait* retries)
                    print(f"{instrument} retry "f"{retries}/"f"{max_retries}")
                    print(e)
                    print(f"Waiting "f"{wait_time}s...")
                    time.sleep(wait_time)

            # failed after retries
            if retries == max_retries:
                print(f"{instrument} chunk failed.")
                chunk_failed = True
                break

            payload = data.get("payload", {})
            rows = payload.get("data", [])
            metadata = payload.get("metadata", {})
            page_count += 1
            total_rows += len(rows)

            print(f"{instrument} page "f"{page_count}"f" | "f"{len(rows)} rows"f" | "f"Total: "f"{total_rows}")

            # PROCESS ROWS
            for row in rows:
                bids = row.get("bid", [])
                asks = row.get("ask", [])
                if (not bids or not asks):
                    continue
                try:
                    best_bid = safe_float(bids[0].get("price"))
                    best_ask = safe_float(asks[0].get("price"))
                    if (best_bid is None or best_ask is None):continue
                    spread = (best_ask - best_bid)
                    mid_price = (best_bid + best_ask) / 2
                    spread_pct = (spread/ mid_price if mid_price != 0 else None)
                    bid_depth_top5 = calculate_depth(bids,5)
                    ask_depth_top5 = calculate_depth(asks,5)
                    bid_depth_top10 = calculate_depth(bids,10)
                    ask_depth_top10 = calculate_depth(asks,10)
                    order_book_imbalance_top5 = calculate_imbalance(bid_depth_top5,ask_depth_top5)
                    order_book_imbalance_top10 = calculate_imbalance(bid_depth_top10,ask_depth_top10)

                    all_rows.append({"instrument":instrument,
                                     "exchange":exchange,
                                     "exchangeTimestamp":row.get("exchangeTimestamp"),
                                     "best_bid":best_bid,
                                     "best_ask":best_ask,
                                     "spread":spread,
                                     "spread_pct":spread_pct,
                                     "bid_depth_top5":bid_depth_top5,
                                     "ask_depth_top5":ask_depth_top5,
                                     "bid_depth_top10":bid_depth_top10,
                                     "ask_depth_top10":ask_depth_top10,
                                     "order_book_imbalance_top5":order_book_imbalance_top5,
                                     "order_book_imbalance_top10":order_book_imbalance_top10})

            next_url = metadata.get("next")
            params = None

        checkpoint_df = pd.DataFrame(all_rows)
        if not checkpoint_df.empty:
            checkpoint_df = (checkpoint_df.drop_duplicates())
            checkpoint_df.to_parquet(checkpoint_file,index=False)
            print(f"{instrument} checkpoint saved: "f"{len(checkpoint_df)} rows")
        else:
            print(f"No rows collected yet for {instrument}.")
        if chunk_failed:
            print(f"Moving to next {instrument} chunk...")
        current_start = current_end

    # FINAL DATAFRAME FOR THIS INSTRUMENT
    orderbook_df = pd.DataFrame(all_rows)

    if orderbook_df.empty:
        print(f"No data collected for {instrument}.")
        return

    orderbook_df = (orderbook_df.drop_duplicates())
    orderbook_df["exchangeTimestamp"] = pd.to_datetime(orderbook_df["exchangeTimestamp"],utc=True)
    orderbook_df = (orderbook_df.sort_values("exchangeTimestamp").reset_index(drop=True))

    print(f"\nFinal shape for {instrument}:")
    print(orderbook_df.shape)
    print(orderbook_df.head())

    # Save outputs
    orderbook_df.to_parquet(final_parquet_file,index=False)
    orderbook_df.to_csv(final_csv_file,index=False)
    orderbook_df.to_csv(final_csv_file,index=False)
    print(f"\nFinished {instrument}.")
    print(f"Saved: {final_parquet_file}")
    print(f"Saved: {final_csv_file}")

# Run script
for instrument in instruments:
    extract_orderbook_features(instrument)

print("\nAll instruments finished.")


######################################################################
Starting extraction for LTCUSDT
######################################################################

LTCUSDT chunk: 2025-07-01 00:00:00+00:00 → 2025-07-04 00:00:00+00:00
LTCUSDT page 1 | 517 rows | Total: 517
LTCUSDT page 2 | 517 rows | Total: 1034
LTCUSDT page 3 | 517 rows | Total: 1551
LTCUSDT page 4 | 517 rows | Total: 2068
LTCUSDT page 5 | 517 rows | Total: 2585
LTCUSDT page 6 | 517 rows | Total: 3102
LTCUSDT page 7 | 517 rows | Total: 3619
LTCUSDT page 8 | 517 rows | Total: 4136
LTCUSDT page 9 | 184 rows | Total: 4320
LTCUSDT checkpoint saved: 4320 rows

LTCUSDT chunk: 2025-07-04 00:00:00+00:00 → 2025-07-07 00:00:00+00:00
LTCUSDT page 1 | 517 rows | Total: 517
LTCUSDT page 2 | 517 rows | Total: 1034
LTCUSDT page 3 | 517 rows | Total: 1551
LTCUSDT page 4 | 517 rows | Total: 2068
LTCUSDT page 5 | 517 rows | Total: 2585
LTCUSDT page 6 | 517 rows | Total: 3102
LTCUSDT page 7 | 517 rows | Total: 3619
LTCUSDT pag

In [1]:
import time
import requests
import pandas as pd
from datetime import datetime, timedelta, UTC
from pathlib import Path

# Configuration
API_KEY = "" # insert key here
BASE_URL = ("https://api.amberdata.com/"
            "markets/futures/order-book-snapshots")
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT"]

headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br"}
BACKFILL_START_DATE = datetime(2025, 7, 1,tzinfo=UTC)
END_DATE = datetime.now(UTC)
CHUNK_DAYS = 3
OVERLAP_MINUTES = 60
MAX_RETRIES = 5
BASE_RETRY_WAIT = 10
output_dir = Path.cwd()

# Functions
def get_symbol_name(instrument):
    return instrument.replace("USDT", "").lower()

def get_output_files(instrument):
    symbol = get_symbol_name(instrument)
    checkpoint_file = output_dir / f"{symbol}_orderbook_features_checkpoint.parquet"
    final_parquet_file = output_dir / f"{symbol}_orderbook_features_1m.parquet"
    final_csv_file = output_dir / f"{symbol}_orderbook_features_1m.csv"
    return final_parquet_file, final_csv_file, checkpoint_file

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None


def calculate_depth(levels, top_n):
    depth = 0.0
    for level in levels[:top_n]:
        price = safe_float(level.get("price"))
        volume = safe_float(level.get("volume"))
        if price is not None and volume is not None:
            depth += price * volume
    return depth


def calculate_imbalance(bid_depth, ask_depth):
    total_depth = bid_depth + ask_depth
    if total_depth == 0:
        return None
    return (bid_depth - ask_depth) / total_depth

def load_existing_data(final_parquet_file, checkpoint_file):
    if final_parquet_file.exists():
        existing_df = pd.read_parquet(final_parquet_file)
    elif checkpoint_file.exists():
        existing_df = pd.read_parquet(checkpoint_file)
    else:
        return pd.DataFrame()

    if existing_df.empty:
        return pd.DataFrame()

    timestamp_col = ("exchangeTimestamp" if "exchangeTimestamp" in existing_df.columns else "timestamp")
    existing_df["exchangeTimestamp"] = pd.to_datetime(existing_df[timestamp_col],utc=True,errors="coerce")

    numeric_columns = ["best_bid",
                       "best_ask",
                       "spread",
                       "spread_pct",
                       "bid_depth_top5",
                       "ask_depth_top5",
                       "bid_depth_top10",
                       "ask_depth_top10",
                       "order_book_imbalance_top5",
                       "order_book_imbalance_top10"]

    for col in numeric_columns:
        if col in existing_df.columns:
            existing_df[col] = pd.to_numeric(existing_df[col],errors="coerce")

    existing_df = (existing_df.dropna(subset=["exchangeTimestamp"]).drop_duplicates(subset=["exchangeTimestamp"], keep="last").sort_values("exchangeTimestamp").reset_index(drop=True))
    return existing_df


def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE

    last_timestamp = existing_df["exchangeTimestamp"].max()
    incremental_start = (last_timestamp.to_pydatetime() - timedelta(minutes=OVERLAP_MINUTES))
    return incremental_start

def parse_orderbook_rows(rows, instrument):
    parsed_rows = []

    for row in rows:
        bids = row.get("bid", [])
        asks = row.get("ask", [])

        if not bids or not asks:
            continue

        best_bid = safe_float(bids[0].get("price"))
        best_ask = safe_float(asks[0].get("price"))

        if best_bid is None or best_ask is None:
            continue

        spread = best_ask - best_bid
        mid_price = (best_bid + best_ask) / 2

        if mid_price == 0:
            spread_pct = None
        else:
            spread_pct = spread / mid_price

        bid_depth_top5 = calculate_depth(bids, 5)
        ask_depth_top5 = calculate_depth(asks, 5)
        bid_depth_top10 = calculate_depth(bids, 10)
        ask_depth_top10 = calculate_depth(asks, 10)
        order_book_imbalance_top5 = calculate_imbalance(bid_depth_top5,ask_depth_top5)
        order_book_imbalance_top10 = calculate_imbalance(bid_depth_top10,ask_depth_top10)

        parsed_rows.append({"instrument": instrument,
                            "exchange": exchange,
                            "exchangeTimestamp": row.get("exchangeTimestamp"),
                            "best_bid": best_bid,
                            "best_ask": best_ask,
                            "spread": spread,
                            "spread_pct": spread_pct,
                            "bid_depth_top5": bid_depth_top5,
                            "ask_depth_top5": ask_depth_top5,
                            "bid_depth_top10": bid_depth_top10,
                            "ask_depth_top10": ask_depth_top10,
                            "order_book_imbalance_top5": order_book_imbalance_top5,
                            "order_book_imbalance_top10": order_book_imbalance_top10})

    parsed_df = pd.DataFrame(parsed_rows)

    if parsed_df.empty:
        return pd.DataFrame()

    parsed_df["exchangeTimestamp"] = pd.to_datetime(parsed_df["exchangeTimestamp"],utc=True,errors="coerce")
    numeric_columns = ["best_bid",
                       "best_ask",
                       "spread",
                       "spread_pct",
                       "bid_depth_top5",
                       "ask_depth_top5",
                       "bid_depth_top10",
                       "ask_depth_top10",
                       "order_book_imbalance_top5",
                       "order_book_imbalance_top10"]

    for col in numeric_columns:
        parsed_df[col] = pd.to_numeric(parsed_df[col],errors="coerce")
    parsed_df = (parsed_df.dropna(subset=["exchangeTimestamp"]).drop_duplicates(subset=["exchangeTimestamp"], keep="last").sort_values("exchangeTimestamp").reset_index(drop=True))
    return parsed_df


def request_page(next_url, params):
    retries = 0

    while retries < MAX_RETRIES:
        try:
            response = requests.get(next_url,headers=headers,params=params,timeout=60)

            # Non-retryable errors
            if response.status_code == 400:
                print("400 Bad Request")
                print("URL:", response.url)
                print("Response:", response.text)
                return None

            if response.status_code == 401:
                print("401 Unauthorized")
                print("Response:", response.text)
                return None

            if response.status_code == 403:
                print("403 Forbidden")
                print("Response:", response.text)
                return None

            if response.status_code == 404:
                print("404 Not Found")
                print("URL:", response.url)
                print("Response:", response.text)
                return None

            if response.status_code == 410:
                print("410 Gone")
                print("URL:", response.url)
                print("Response:", response.text)
                return None

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            retries += 1
            wait_time = BASE_RETRY_WAIT * retries
            print(f"Retry {retries}/{MAX_RETRIES}")
            print(e)
            print(f"Waiting {wait_time}s")
    return None

def fetch_orderbook_chunk(instrument,current_start,current_end):
    next_url = f"{BASE_URL}/{instrument}"
    params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat": "iso8601"}
    page_count = 0
    total_rows = 0
    chunk_dfs = []

    while next_url:
        data = request_page(next_url=next_url,params=params)

        if data is None:
            print(f"{instrument} chunk failed.")
            break

        payload = data.get("payload", {})
        rows = payload.get("data", [])
        metadata = payload.get("metadata", {})

        page_count += 1
        total_rows += len(rows)

        print(f"{instrument} page {page_count} "f"| {len(rows):,} rows "f"| total raw rows: {total_rows:,}")
        parsed_df = parse_orderbook_rows(rows=rows,instrument=instrument)

        if not parsed_df.empty:
            chunk_dfs.append(parsed_df)

        next_url = metadata.get("next")
        params = None

    if not chunk_dfs:
        return pd.DataFrame()

    chunk_df = pd.concat(chunk_dfs,ignore_index=True)
    chunk_df = (chunk_df.drop_duplicates(subset=["exchangeTimestamp"], keep="last").sort_values("exchangeTimestamp").reset_index(drop=True))
    print(f"{instrument} parsed rows in chunk: {len(chunk_df):,}")
    return chunk_df


def pull_new_orderbook_data(instrument,current_start,end_date):
    all_chunks = []

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS),end_date)

        print("\n" + "=" * 50)
        print(f"{instrument} chunk: "f"{current_start} → {current_end}")
        chunk_df = fetch_orderbook_chunk(instrument=instrument,current_start=current_start,current_end=current_end)

        if not chunk_df.empty:
            all_chunks.append(chunk_df)
        current_start = current_end


    if not all_chunks:
        return pd.DataFrame()

    new_df = pd.concat(all_chunks,ignore_index=True)
    new_df = (new_df.drop_duplicates(subset=["exchangeTimestamp"], keep="last").sort_values("exchangeTimestamp").reset_index(drop=True))
    return new_df

def finalize_orderbook_data(orderbook_df,instrument):
    if orderbook_df.empty:
        return orderbook_df

    orderbook_df = orderbook_df.copy()
    timestamp_col = ("exchangeTimestamp" if "exchangeTimestamp" in orderbook_df.columns else "timestamp")
    orderbook_df["exchangeTimestamp"] = pd.to_datetime(orderbook_df[timestamp_col],utc=True,errors="coerce")
    orderbook_df["instrument"] = instrument
    orderbook_df["exchange"] = exchange

    numeric_columns = ["best_bid",
                       "best_ask",
                       "spread",
                       "spread_pct",
                       "bid_depth_top5",
                       "ask_depth_top5",
                       "bid_depth_top10",
                       "ask_depth_top10",
                       "order_book_imbalance_top5",
                       "order_book_imbalance_top10"]

    for col in numeric_columns:
        if col not in orderbook_df.columns:
            orderbook_df[col] = pd.NA
        orderbook_df[col] = pd.to_numeric(orderbook_df[col],errors="coerce")
    orderbook_df = (orderbook_df.dropna(subset=["exchangeTimestamp"]).drop_duplicates(subset=["exchangeTimestamp"], keep="last").sort_values("exchangeTimestamp").reset_index(drop=True))
    orderbook_df = orderbook_df[["exchangeTimestamp",
                                 "instrument",
                                 "exchange",
                                 "best_bid",
                                 "best_ask",
                                 "spread",
                                 "spread_pct",
                                 "bid_depth_top5",
                                 "ask_depth_top5",
                                 "bid_depth_top10",
                                 "ask_depth_top10",
                                 "order_book_imbalance_top5",
                                 "order_book_imbalance_top10"]]
    return orderbook_df

# ============================================================
# EXTRACTION
# ============================================================

def extract_orderbook_features(instrument):
    final_parquet_file, final_csv_file, checkpoint_file = get_output_files(instrument)

    print("######################################################################")
    print(f"Starting orderbook extraction for {instrument}")
    print("######################################################################")

    existing_df = load_existing_data(final_parquet_file=final_parquet_file,checkpoint_file=checkpoint_file)

    current_start = determine_start_date(existing_df)
    print(f"Existing rows: {len(existing_df):,}")

    if not existing_df.empty:
        print(f"Existing latest timestamp: "f"{existing_df['exchangeTimestamp'].max()}")

    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")

    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return existing_df

    new_df = pull_new_orderbook_data(instrument=instrument,current_start=current_start,end_date=END_DATE)

    if new_df.empty and existing_df.empty:
        print(f"No orderbook data for {instrument}. Skipping save.")
        return None

    if new_df.empty:
        print("\nNo new rows fetched.")
        combined_df = existing_df.copy()
    else:
        print(f"\nNew rows before merge: {len(new_df):,}")
        combined_df = pd.concat([existing_df, new_df],ignore_index=True)
    orderbook_df = finalize_orderbook_data(orderbook_df=combined_df,instrument=instrument)

    if orderbook_df.empty:
        print(f"No valid orderbook rows for {instrument}. Skipping save.")
        return None

    # export results
    orderbook_df.to_parquet(checkpoint_file,index=False)
    orderbook_df.to_parquet(final_parquet_file,index=False)
    orderbook_df.to_csv(final_csv_file,index=False)
    print(f"\nFinal shape for {instrument}: {orderbook_df.shape}")
    print(f"Final earliest timestamp: {orderbook_df['exchangeTimestamp'].min()}")
    print(f"Final latest timestamp: {orderbook_df['exchangeTimestamp'].max()}")
    print(orderbook_df.head())
    print(f"Saved parquet: {final_parquet_file}")
    print(f"Saved csv: {final_csv_file}")
    print(f"Saved checkpoint: {checkpoint_file}")
    print(f"Finished {instrument}")
    return orderbook_df

# Run script
results = {}
for instrument in instruments:
    df = extract_orderbook_features(instrument)
    if df is not None:
        results[instrument] = df

print("\nAll instruments finished.")
print("Successful instruments:", list(results.keys()))


######################################################################
Starting orderbook extraction for LTCUSDT
######################################################################
Existing rows: 488,152
Existing latest timestamp: 2026-06-05 00:34:00.899000+00:00
Incremental start: 2026-06-04 23:34:00.899000+00:00
End date: 2026-06-12 20:46:47.431318+00:00

LTCUSDT chunk: 2026-06-04 23:34:00.899000+00:00 → 2026-06-07 23:34:00.899000+00:00
LTCUSDT page 1 | 517 rows | total raw rows: 517
LTCUSDT page 2 | 517 rows | total raw rows: 1,034
LTCUSDT page 3 | 517 rows | total raw rows: 1,551
LTCUSDT page 4 | 518 rows | total raw rows: 2,069
LTCUSDT page 5 | 517 rows | total raw rows: 2,586
LTCUSDT page 6 | 517 rows | total raw rows: 3,103
LTCUSDT page 7 | 517 rows | total raw rows: 3,620
LTCUSDT page 8 | 517 rows | total raw rows: 4,137
LTCUSDT page 9 | 183 rows | total raw rows: 4,320
LTCUSDT parsed rows in chunk: 4,320

LTCUSDT chunk: 2026-06-07 23:34:00.899000+00:00 → 2026-06-10 23:34:0